In [1]:
# ========== 导入：第 4 周「代码注释助手」要用的库 ==========

# os：读环境变量里的各家 API Key
import os
# io：本练习导入备用（字符串缓冲等）；与后续扩展相关
import io
# sys：标准输入输出相关（本格主要作通用导入）
import sys
# load_dotenv：从 .env 加载密钥，避免写死在笔记本里
from dotenv import load_dotenv
# OpenAI 客户端：也可指向 Ollama / OpenRouter 的 OpenAI 兼容接口
from openai import OpenAI
# gradio：搭左右对照的代码注释 Web UI
import gradio as gr
# subprocess：子进程相关（本练习主流程未必用到，保留原导入）
import subprocess
# IPython 展示工具：Markdown / display（本格导入备用）
from IPython.display import Markdown, display


In [ ]:
# ========== 环境变量：加载并检查多家 API Key 是否存在 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# OpenAI 官方密钥
openai_api_key = os.getenv('OPENAI_API_KEY')
# Anthropic（可选）
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
# Google（可选）
google_api_key = os.getenv('GOOGLE_API_KEY')
# Grok / xAI（可选）
grok_api_key = os.getenv('GROK_API_KEY')
# Groq（可选）
groq_api_key = os.getenv('GROQ_API_KEY')
# OpenRouter：本练习多模型路由常用（可选但推荐）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 以下 print 文案是运行时状态提示，保留英文原样
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [3]:
# ========== 连接客户端：本地 Ollama + 云端 OpenRouter（OpenAI 兼容） ==========

# 本地 Ollama 的 OpenAI 兼容基址（需本机已启动 ollama serve）
ollama_url = "http://localhost:11434/v1"
# OpenRouter 的 OpenAI 兼容 API 基址（URL 禁止改译）
openrouter_url = "https://openrouter.ai/api/v1"

# api_key 对本地 Ollama 常可随意填；真正鉴权看服务端配置
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
# OpenRouter：用上一格读到的 OPENROUTER_API_KEY
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [4]:
# ========== 模型清单 + 路由表：哪个 model id 走哪个客户端 ==========

# 下拉可选的模型 id 列表（字符串原样；含 OpenRouter 与 Ollama 本地名）
models = ["openai/gpt-5.2", "anthropic/claude-sonnet-4.5", "x-ai/grok-4", "gemini-2.5-pro", "qwen3:4b", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b:free", ]

# model id → 客户端：云端走 openrouter，本地 tag 走 ollama
clients = {"openai/gpt-5.2": openrouter, "anthropic/claude-sonnet-4.5": openrouter, "x-ai/grok-4": openrouter, "gemini-2.5-pro": openrouter, "openai/gpt-oss-120b:free": openrouter, "qwen3:4b": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}


In [5]:
# ========== System 指令：只加注释/文档字符串，禁止改逻辑（prompt 原文保留） ==========

# instruction：发给模型的 system prompt；整段英文是可运行行为字符串，禁止改译
instruction = """
You are a code documentation assistant. Your sole task is to add clear, concise comments and docstrings to code provided by the user.

Instructions:
1. Greet the user with exactly: "Paste your code below and I'll add comments and docstrings to it."
2. When the user pastes code, automatically detect the programming language.
3. Return the EXACT same code — same logic, same structure, same variable names, same formatting — with only comments and docstrings added. Do NOT modify, refactor, optimize, or fix any part of the code.
4. Use the idiomatic comment and docstring style for the detected language:
   - Python: # for inline comments, \"\"\"triple-quote docstrings\"\"\" for functions/classes/modules
   - JavaScript/TypeScript: // for inline comments, /** JSDoc */ for functions/classes
   - Go: // for inline comments, // preceding function signature for godoc
   - Java/Kotlin: // for inline comments, /** Javadoc */ for classes/methods
   - Rust: // for inline comments, /// doc comments for functions/structs
   - C/C++: // for inline comments, /** Doxygen */ for functions/structs
   - Ruby: # for inline comments, yard-style comments for methods/classes
   - For any other language, follow its established documentation conventions.
5. Comment coverage:
   - Add a top-level docstring/comment summarizing the file or snippet's purpose.
   - Add docstrings to every function, method, and class describing what it does, its parameters, and its return value.
   - Add inline comments only where the logic is non-obvious — do NOT narrate every line.
6. Output ONLY the commented code inside a single fenced code block tagged with the detected language. No explanations, no preamble, no follow-up text outside the code block.
"""


In [ ]:
# ========== 核心函数：选模型 → chat.completions → 剥掉 markdown 围栏 ==========

# re：用正则去掉模型常爱加的 ```lang 围栏
import re

def comment_code(model, code):
    # 按下拉选中的 model id 取对应客户端（OpenRouter 或 Ollama）
    client = clients[model]
    # Chat Completions：system=注释规范，user=用户粘贴的代码
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": code}
        ]
    )
    # 取出助手回复正文
    reply = response.choices[0].message.content
    # 去掉开头的 ```xxx\n（若有）
    reply = re.sub(r"^```\w*\n", "", reply)
    # 去掉结尾的 \n```（若有）
    reply = re.sub(r"\n```$", "", reply)
    # 返回干净代码文本，填回 Gradio 输出框
    return reply


In [ ]:
# ========== Gradio UI：左贴代码、右看注释结果，选模型后一键 Add Comments ==========

# 从同目录 styles 导入 CSS（外部样式文件；import 名保持原样）
from styles import CSS

# Blocks：自定义 css + Monochrome 主题；title 是浏览器标签文案
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Code Commenter") as ui:
    # 顶部提示（UI 字符串原样保留）
    gr.Markdown("## Paste your code below and I'll add comments and docstrings to it.")
    # 左右等宽两列：输入 / 输出
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            # 左侧：用户粘贴原始代码
            code_input = gr.Code(
                label="Your code",
                value="",
                lines=26
            )
        with gr.Column(scale=6):
            # 右侧：模型返回的带注释代码
            code_output = gr.Code(
                label="Commented code",
                value="",
                lines=26
            )

    # 底部控件行：模型下拉 + 按钮（CSS class 名保持原样）
    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        comment_btn = gr.Button("Add Comments", elem_classes=["convert-btn"])

    # 点击：comment_code(model, code_input) → code_output
    comment_btn.click(fn=comment_code, inputs=[model, code_input], outputs=[code_output])

# 启动界面并尝试打开浏览器
ui.launch(inbrowser=True)
